# Two-Stage YOLO11s Transfer Learning for Hand Finger-Number Detection (1–5)

This notebook reproduces the full training and evaluation pipeline for the YOLO11s model served by the FastAPI application:

1. **Stage 1**: Fine-tune `yolo11s.pt` on Roboflow **Fingers Numbers v1** (1,152 images, classes `0–5`, CC BY 4.0).
2. **Stage 2 (Transfer Learning)**: Initialize from Stage 1 best weights and fine-tune on Roboflow **1-5 v3** (750 images, classes `1–5`, CC BY 4.0).
3. **Evaluation & Visualization**: Compute validation/test mAP metrics and compare ground-truth vs predicted bounding boxes.
4. **Export**: Copy the final `best.pt` checkpoint into `weights/best.pt` for the FastAPI server.

### Prerequisites
Install `requirements.txt` and `requirements-extras.txt`, and export your Roboflow API key:
```bash
export ROBOFLOW_API_KEY="your_key_here"   # PowerShell: $env:ROBOFLOW_API_KEY="your_key_here"
```

In [ ]:
import os
import shutil
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from roboflow import Roboflow
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

# Load optional local .env file without overriding existing environment variables.
env_file = PROJECT_ROOT / ".env"
if env_file.is_file():
    for raw_line in env_file.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"\' '))

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Download Stage 1 Dataset (Roboflow — Fingers Numbers v1, CC BY 4.0)

In [ ]:
api_key = os.getenv("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Set the ROBOFLOW_API_KEY environment variable before downloading datasets.")

rf = Roboflow(api_key=api_key)
stage1_dataset = (
    rf.workspace("wyde")
    .project("fingers-numbers-mobih")
    .version(1)
    .download("yolov11", location=str(DATASETS_DIR / "Fingers-Numbers-1"))
)
stage1_data_yaml = Path(stage1_dataset.location) / "data.yaml"
print("Stage 1 data.yaml:", stage1_data_yaml)

## 2. Stage 1 Training & Evaluation (`YOLO11s` Pretrained on COCO)

In [ ]:
stage1_model = YOLO("yolo11s.pt")
stage1_results = stage1_model.train(
    data=str(stage1_data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(PROJECT_ROOT / "runs" / "detect"),
    name="stage1_fingers_0_5",
)
stage1_best = Path(stage1_results.save_dir) / "weights" / "best.pt"
print("Stage 1 best weights:", stage1_best)

In [ ]:
stage1_eval = YOLO(str(stage1_best))

val_metrics = stage1_eval.val(split="val")
print(f"Stage 1 Val  | mAP@50: {val_metrics.box.map50:.4f} | mAP@50-95: {val_metrics.box.map:.4f}")

test_metrics = stage1_eval.val(split="test")
print(f"Stage 1 Test | mAP@50: {test_metrics.box.map50:.4f} | mAP@50-95: {test_metrics.box.map:.4f}")

## 3. Download Stage 2 Dataset (Roboflow — 1-5 v3, CC BY 4.0)

In [ ]:
stage2_dataset = (
    rf.workspace("wyde")
    .project("1-5-y1dwp-muxo0")
    .version(3)
    .download("yolov11", location=str(DATASETS_DIR / "1-5-3"))
)
stage2_data_yaml = Path(stage2_dataset.location) / "data.yaml"
print("Stage 2 data.yaml:", stage2_data_yaml)

## 4. Stage 2 Transfer Learning (`1–5` Hand Gestures)

In [ ]:
# Initialize from Stage 1 weights (backbone + neck transfer) and adapt head to 5 classes (1-5).
stage2_model = YOLO("yolo11s.yaml").load(str(stage1_best))
stage2_results = stage2_model.train(
    data=str(stage2_data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(PROJECT_ROOT / "runs" / "detect"),
    name="stage2_fingers_1_5",
)
stage2_best = Path(stage2_results.save_dir) / "weights" / "best.pt"
print("Stage 2 best weights:", stage2_best)

In [ ]:
final_model = YOLO(str(stage2_best))

val_s2 = final_model.val(split="val")
print(f"Stage 2 Val  | mAP@50: {val_s2.box.map50:.4f} | mAP@50-95: {val_s2.box.map:.4f}")
print("Per-class mAP@50-95 (val):", dict(zip(final_model.names.values(), val_s2.box.maps.round(4))))

test_s2 = final_model.val(split="test")
print(f"Stage 2 Test | mAP@50: {test_s2.box.map50:.4f} | mAP@50-95: {test_s2.box.map:.4f}")

## 5. Visual Comparison: Ground Truth vs Predicted Bounding Boxes

In [ ]:
def plot_gt_vs_pred(model: YOLO, image_paths: list[Path]) -> None:
    """Plot original images, ground-truth labels, and model predictions side by side."""
    predictions = model.predict([str(p) for p in image_paths], verbose=False)
    class_names = model.names

    fig, axes = plt.subplots(3, len(image_paths), figsize=(6 * len(image_paths), 14))
    if len(image_paths) == 1:
        axes = axes[:, np.newaxis]

    for col, (img_path, result) in enumerate(zip(image_paths, predictions)):
        orig_img = Image.open(img_path)
        width, height = orig_img.size
        label_path = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"

        axes[0, col].imshow(orig_img)
        axes[0, col].set_title(f"Original: {img_path.name}")
        axes[0, col].axis("off")

        axes[1, col].imshow(orig_img)
        axes[1, col].set_title("Ground Truth")
        axes[1, col].axis("off")
        if label_path.is_file():
            for line in label_path.read_text(encoding="utf-8").splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1 = (xc - bw / 2) * width
                    y1 = (yc - bh / 2) * height
                    rect = patches.Rectangle((x1, y1), bw * width, bh * height, linewidth=2, edgecolor="lime", facecolor="none")
                    axes[1, col].add_patch(rect)
                    axes[1, col].text(x1, max(12, y1 - 8), f"Label: {class_names.get(cls_id, cls_id)}", color="white", fontsize=9, bbox=dict(facecolor="green", alpha=0.7))

        axes[2, col].imshow(orig_img)
        axes[2, col].set_title("Predicted")
        axes[2, col].axis("off")
        for box, conf, cls_raw in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
            x1, y1, x2, y2 = box.cpu().tolist()
            cls_id = int(cls_raw.item())
            label = class_names.get(cls_id, str(cls_id))
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
            axes[2, col].add_patch(rect)
            axes[2, col].text(x1, max(12, y1 - 8), f"Pred: {label} ({conf.item():.2f})", color="white", fontsize=9, bbox=dict(facecolor="red", alpha=0.7))

    plt.tight_layout()
    plt.show()


sample_images = sorted((DATASETS_DIR / "1-5-3" / "test" / "images").glob("*.jpg"))[:2]
if sample_images:
    plot_gt_vs_pred(final_model, sample_images)

## 6. Export Final Weights for FastAPI Serving

In [ ]:
target_weights = PROJECT_ROOT / "weights" / "best.pt"
target_weights.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(stage2_best, target_weights)
print(f"Copied trained weights to {target_weights}")

## 7. Optional Local OpenCV Webcam Sanity Check

In [ ]:
# Run this cell interactively on a machine with a connected webcam (press 'q' to quit).
import cv2

def run_local_webcam(weights_path: Path, conf: float = 0.5) -> None:
    webcam_model = YOLO(str(weights_path))
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Unable to open default webcam (index 0).")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            results = webcam_model(frame, imgsz=640, conf=conf, verbose=False)
            annotated = results[0].plot()
            cv2.imshow("YOLO11s Hand Gesture Webcam", annotated)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()

# Uncomment to run locally:
# run_local_webcam(PROJECT_ROOT / "weights" / "best.pt")